# The notebook calculates how much time it takes for a PAS pipeline

In [ ]:
import pandas as pd

# Show all columns and don't truncate long strings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

# Show all rows 
pd.set_option("display.max_rows", None)

In [42]:
import os
import re
import pandas as pd
import numpy as np

log_dir = "slurm_logs"

# Regex patterns
split_pattern = re.compile(r"with split\s+([0-9/]+)")
elapsed_pattern = re.compile(r"\[Elapsed\]\s+([0-9:.]+)")

records = []

for file in os.listdir(log_dir):
    if file.startswith("timer") and file.endswith(".out"):
        filepath = os.path.join(log_dir, file)
        with open(filepath, "r") as f:
            text = f.read()

        # Find all splits and elapsed times
        splits = split_pattern.findall(text)
        elapsed = elapsed_pattern.findall(text)

        # Pad elapsed list so it matches splits (missing ones → NaN)
        while len(elapsed) < len(splits):
            elapsed.append(np.nan)

        # Combine each experiment
        for s, e in zip(splits, elapsed):
            records.append({
                "filename": file,
                "split": s,
                "elapsed": e
            })

# Build DataFrame
df = pd.DataFrame(records)
df = df.dropna(subset = ['elapsed'])
df

,filename,split,elapsed
0,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_10_1334394.out,12/4/800,0:00:26.520580
1,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_10_1334394.out,2400/800/800,0:01:47.531551
2,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_11_1334395.out,12/4/800,0:00:28.070493
3,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_11_1334395.out,2400/800/800,0:01:46.481745
4,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_12_1334396.out,12/4/800,0:00:26.629491
5,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_12_1334396.out,2400/800/800,0:01:49.160280
6,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_13_1334397.out,12/4/800,0:00:26.772897
7,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_13_1334397.out,2400/800/800,0:01:48.794469
8,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_14_1334398.out,12/4/800,0:00:26.408740
9,timerscale_up_DeepSeek-R1-Distill-Llama-8B_seed_14_1334398.out,2400/800/800,0:01:46.653161


In [43]:
import pandas as pd
import numpy as np

# Convert elapsed strings (HH:MM:SS.microseconds) into timedelta
df["elapsed_timedelta"] = pd.to_timedelta(df["elapsed"], errors="coerce")

# Convert to seconds as float
df["elapsed_seconds"] = df["elapsed_timedelta"].dt.total_seconds()

# Example: experiments with '12' in split
mean_fast = df[df['split'].str.contains('12', na=False)]["elapsed_seconds"].mean()
var_fast  = df[df['split'].str.contains('12', na=False)]["elapsed_seconds"].var()

# Experiments without '12'
mean_slow = df[~df['split'].str.contains('12', na=False)]["elapsed_seconds"].mean()
var_slow  = df[~df['split'].str.contains('12', na=False)]["elapsed_seconds"].var()

print("Mean (12/4/800 split):", mean_fast)
print("Variance (12/4/800 split):", var_fast)
print("Mean (2400/800/800 split):", mean_slow)
print("Variance (2400/800/800 split):", var_slow)

Mean (12/4/800 split): 32.805121622222224
Variance (12/4/800 split): 463.53847536602717
Mean (2400/800/800 split): 103.42685363414634
Variance (2400/800/800 split): 15.127384624419985
